# 5. Validação de ciclo

Mesma checagem de `src/lib/craftingGraph.ts` (DFS de 3 estados), rodada
aqui sobre o grafo de recipes antes do export - o objetivo é falhar cedo
no pipeline de dados, em vez de só descobrir um ciclo em runtime no app.

Usa `cg.find_cycles` de `scripts/crafting_graph.py` (mesma função que o
gate do build chama), em vez de reimplementar o DFS aqui.


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
sys.path.insert(0, str(REPO_ROOT / "scripts"))

import crafting_graph as cg  # scripts/crafting_graph.py - the single source of truth


## Montar o grafo de dependências (item -> ingredientes) via o módulo


In [2]:
items = cg.load_rows(cg.ITEM_DT)
recipes = cg.load_rows(cg.RECIPE_DT)
resolve = cg.build_resolver(items)

item_graph, _ = cg.build_item_graph(items, recipes, resolve)
adjacency = {item_id: list(entry["ingredients"]) for item_id, entry in item_graph.items()}
print(f"{len(adjacency)} nós no grafo de itens")


1622 nós no grafo de itens


## DFS de 3 estados (unvisited / in_progress / done)


In [3]:
cycles = cg.find_cycles(adjacency)
print(f"{len(cycles)} ciclo(s) encontrado(s) no grafo real")
cycles


0 ciclo(s) encontrado(s) no grafo real


[]

## Controle positivo

Prova que o detector realmente pega um ciclo, injetando um sintético -
sem isso, "0 ciclos encontrados" acima poderia só significar um detector
quebrado.


In [4]:
test_graph = {"A": ["B"], "B": ["C"], "C": ["A"]}
test_cycles = cg.find_cycles(test_graph)
assert test_cycles, "detector de ciclo não pegou um ciclo sintético óbvio - tem bug no detector"
print("Controle positivo OK:", test_cycles)


Controle positivo OK: [['A', 'B', 'C', 'A']]
